In [1]:
"""import os
import pickle
import pandas as pd
import tensorflow as tf
from google.colab import drive

# 1. Access your files
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/sentiment-analysis-project'
os.chdir(PROJECT_ROOT)

# 2. Load the Model
# Use the same .h5 name you used in Section 3
model = tf.keras.models.load_model("models/lstm_model.h5")

# 3. Load the Tokenizer and Label Encoder
with open("artifacts/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

with open("artifacts/label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

# 4. Load the results from Section 3 to evaluate
df_results = pd.read_csv("results/metrics/aspect_results.csv")

print("✅ Section 4 Loaded: Model and Artifacts are ready for evaluation!")"""

'import os\nimport pickle\nimport pandas as pd\nimport tensorflow as tf\nfrom google.colab import drive\n\n# 1. Access your files\ndrive.mount(\'/content/drive\')\nPROJECT_ROOT = \'/content/drive/MyDrive/sentiment-analysis-project\'\nos.chdir(PROJECT_ROOT)\n\n# 2. Load the Model\n# Use the same .h5 name you used in Section 3\nmodel = tf.keras.models.load_model("models/lstm_model.h5")\n\n# 3. Load the Tokenizer and Label Encoder\nwith open("artifacts/tokenizer.pkl", "rb") as f:\n    tokenizer = pickle.load(f)\n\nwith open("artifacts/label_encoder.pkl", "rb") as f:\n    le = pickle.load(f)\n\n# 4. Load the results from Section 3 to evaluate\ndf_results = pd.read_csv("results/metrics/aspect_results.csv")\n\nprint("✅ Section 4 Loaded: Model and Artifacts are ready for evaluation!")'

# **Stage 4.1 — LIME**

In [2]:
%pip install lime

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# ==============================
# STAGE 4.1 - LIME EXPLAINABILITY
# ==============================


import pandas as pd
import joblib
import numpy as np
from lime.lime_text import LimeTextExplainer

# ------------------------------
# 1. LOAD DATA (for test samples)
# ------------------------------
df = pd.read_csv('data/processed/cleaned_reviews.csv')
df['final_text'] = df['final_text'].fillna('')

X = df['final_text']
y = df['sentiment']

# ------------------------------
# 2. LOAD MODEL + VECTORIZER
# ------------------------------
model = joblib.load('models/best_traditional_model.pkl')
vectorizer = joblib.load('models/tfidf_vectorizer.pkl')

# VERSION FIX: Manually set multi_class attribute to fix scikit-learn version mismatch
if not hasattr(model, 'multi_class'):
    model.multi_class = 'auto'

# ------------------------------
# 3. CREATE LIME EXPLAINER
# ------------------------------
explainer = LimeTextExplainer(
    class_names=['negative', 'neutral', 'positive']
)

# ------------------------------
# 4. PREDICT PROBA FUNCTION
# ------------------------------
def predict_proba(texts):
    vec = vectorizer.transform(texts)
    return model.predict_proba(vec)

# ------------------------------
# 5. GET SAMPLE DATA
# ------------------------------
df_sample = df.sample(100, random_state=42).reset_index(drop=True)
X_sample = df_sample['final_text']
y_sample = df_sample['sentiment']

# ------------------------------
# 6. GET PREDICTIONS
# ------------------------------
X_vec = vectorizer.transform(X_sample)
y_pred = model.predict(X_vec)

# ------------------------------
# 7. FIND CASES
# ------------------------------
correct_idx = np.where(y_pred == y_sample)[0]
incorrect_idx = np.where(y_pred != y_sample)[0]

# ------------------------------
# 8. EXPLAIN ONE CORRECT CASE
# ------------------------------
if len(correct_idx) > 0:
    idx = correct_idx[0]
    text = X_sample.iloc[idx]
    print("\n✅ CORRECT PREDICTION")
    print("Text:", text)
    print("True:", y_sample.iloc[idx])
    print("Pred:", y_pred[idx])
    exp = explainer.explain_instance(text, predict_proba, num_features=10)
    exp.save_to_file('results/figures/lime_correct.html')

# ------------------------------
# 9. EXPLAIN ONE WRONG CASE
# ------------------------------
if len(incorrect_idx) > 0:
    idx = incorrect_idx[0]
    text = X_sample.iloc[idx]
    print("\n❌ WRONG PREDICTION")
    print("Text:", text)
    print("True:", y_sample.iloc[idx])
    print("Pred:", y_pred[idx])
    exp = explainer.explain_instance(text, predict_proba, num_features=10)
    exp.save_to_file('results/figures/lime_wrong.html')

print("\n✅ LIME explanations saved in results/figures/")


✅ CORRECT PREDICTION
Text: agree reviewer color pink person subtle thought thing pro pocket make fun chic overall style material lovely breathable find sheer course wear outside dressing room material thicker hang body well run mostly tt tried medium fit everywhere slightly tighter chest
True: positive
Pred: positive

❌ WRONG PREDICTION
Text: send back exchange way big excited get smaller size really beautifully made
True: neutral
Pred: negative

✅ LIME explanations saved in results/figures/


# **Stage 4.2 — Streamlit App**

In [9]:
%pip install streamlit

%pip install plotly


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -------------------- ------------------- 5.0/9.9 MB 22.1 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 22.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
"""import os

def list_files(startpath):
    for root, dirs, files in os.walk(startpath):
        # Exclude hidden folders like .ipynb_checkpoints
        dirs[:] = [d for d in dirs if not d.startswith('.')]

        level = root.replace(startpath, '').count(os.sep)
        indent = ' ' * 4 * (level)
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if not f.startswith('.'):
                print(f'{subindent}{f}')

# Change this to your actual Drive folder name
project_path = '/content/drive/MyDrive/sentiment-analysis-project'

print(f"Structure for: {project_path}\n")
list_files(project_path)"""

'import os\n\ndef list_files(startpath):\n    for root, dirs, files in os.walk(startpath):\n        # Exclude hidden folders like .ipynb_checkpoints\n        dirs[:] = [d for d in dirs if not d.startswith(\'.\')]\n\n        level = root.replace(startpath, \'\').count(os.sep)\n        indent = \' \' * 4 * (level)\n        print(f\'{indent}{os.path.basename(root)}/\')\n        subindent = \' \' * 4 * (level + 1)\n        for f in files:\n            if not f.startswith(\'.\'):\n                print(f\'{subindent}{f}\')\n\n# Change this to your actual Drive folder name\nproject_path = \'/content/drive/MyDrive/sentiment-analysis-project\'\n\nprint(f"Structure for: {project_path}\n")\nlist_files(project_path)'

In [7]:
streamlit --version

NameError: name 'streamlit' is not defined